# Proyecto 3: Histograma de Temperaturas y MergeSort con CUDA

**Universidad del Valle de Guatemala**  
**CC3069 - Computación Paralela y Distribuida** — Semestre II, 2026  
**Equipo:**  
- Marinés García (23391)  
- Camila Richter (23183)  
- Carlos Alburez (231311)  

---  
### Resumen de la solución
Este proyecto optimiza y acelera el algoritmo de la **Evaluación Corta 2** (MergeSort + Histograma de 100 cubetas) mediante una arquitectura híbrida CPU-GPU:
1. **Fase CPU multinúcleo (`multiprocessing`):**  
   - F1: Generación determinista de temperaturas por bloques con semillas independientes.  
   - F6: Formateo y escritura paralela a disco de `temperaturas_ordenadas.csv`, `histograma.dat` e `histograma.png`.
2. **Fase GPU masiva (`Numba CUDA`):**  
   - F2/F5: Transferencias Host $\leftrightarrow$ Device a través de PCIe.  
   - F3: **MergeSort bottom-up iterativo**, alternando buffers en VRAM sin transferencias intermedias.  
   - F4: **Histograma con memoria compartida (Shared Memory)** y sumas atómicas locales por bloque.

## 1. Verificación del Entorno y GPU
Se comprueba la disponibilidad de la GPU NVIDIA (T4 en Google Colab) y la compatibilidad con CUDA.

In [ ]:
# Clonar y entrar al repositorio en Google Colab si aún no estamos dentro
import os
if not os.path.exists("comun"):
    !git clone https://github.com/NESHGP04/proyecto3-PL.git
    %cd proyecto3-PL

!nvidia-smi

In [ ]:
from numba import cuda
print("Dispositivo CUDA detectado:")
cuda.detect()

## 2. Instalación de Dependencias e Importaciones

In [ ]:
!pip install -q -r requirements.txt

import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

from comun.config import TAMANOS, CUBETAS, FASES
from comun.datos import generar_secuencial
import secuencial.histograma_seq as seq
import paralelo.histograma_cuda as cuda_pipe

## 3. Pruebas de Validación y Corrección Algorítmica
Antes de medir rendimiento, se valida obligatoriamente que:
- El arreglo ordenado por la GPU sea idéntico a `np.sort`.
- El histograma de la GPU coincida exactamente con la versión secuencial de referencia.
- La suma total de frecuencias en las 100 cubetas sea estrictamente igual a $N$.

In [ ]:
# Validación con N = 100,000 elementos
cuda_pipe.validar(100_000)

## 4. Ejecución del Pipeline Paralelo con Monitoreo de Recursos
Se ejecuta una corrida de demostración con $N = 1,000,000$ registrando el uso de recursos de hardware en segundo plano.

In [ ]:
from benchmarks.monitor_recursos import MonitorCPU, guardar_estado_gpu

guardar_estado_gpu("evidencias/nvidia_smi_antes.txt")

print("Corriendo pipeline paralelo con N = 1,000,000 y monitoreo de CPU activo...")
with MonitorCPU("evidencias/cpu_por_nucleo.csv", intervalo=0.2):
    ordenado, hist, tiempos = cuda_pipe.ejecutar(1_000_000, escribir_archivos=True, carpeta_salida="resultados/demo")

guardar_estado_gpu("evidencias/nvidia_smi_durante.txt")

print("\n--- Desglose de tiempos de la versión CUDA (N = 1,000,000) ---")
for fase in FASES:
    print(f"{fase:<15}: {tiempos[fase]:.5f} s")

## 5. Benchmarks Oficiales (10 repeticiones por tamaño)
Se ejecuta la suite completa de benchmarks para $N = 10^4, 10^5, 10^6$ (descartando la primera corrida como calentamiento JIT).

In [ ]:
!python -m benchmarks.run_benchmarks --modo cuda

Se generan las tablas comparativas, métricas de speedup por fase y gráficas oficiales:

In [ ]:
!python -m benchmarks.run_benchmarks --modo graficas

## 6. Resultados y Visualización de Gráficas de Speedup

In [ ]:
figuras = [
    "informe/figuras/comparativa_total.png",
    "informe/figuras/speedup_por_fase.png",
    "informe/figuras/cuda_porcentaje_por_fase.png"
]

for ruta in figuras:
    if os.path.exists(ruta):
        print(f"\nMostrando {ruta}:")
        display(Image(filename=ruta))
    else:
        print(f"Gráfica {ruta} aún no generada (ejecutar benchmarks primero).")

## 7. Tabla Comparativa de Rendimiento y Speedup

In [ ]:
tabla_md = "resultados/tabla_comparativa.md"
if os.path.exists(tabla_md):
    with open(tabla_md, "r", encoding="utf-8") as f:
        print(f.read())
else:
    print("Tabla comparativa no encontrada. Asegúrese de correr --modo graficas con ambos resultados.")

## 8. Compilación del Informe Final en HTML / PDF

In [ ]:
!pip install -q markdown
!python informe/armar_informe.py

print("\nDescarga del informe compilado:")
try:
    from google.colab import files
    if os.path.exists("informe/Informe_Proyecto3.html"):
        files.download("informe/Informe_Proyecto3.html")
        print("Descargando informe/Informe_Proyecto3.html... Ábrelo en Chrome/Safari y presiona Imprimir -> Guardar como PDF.")
except ImportError:
    pass